# Differentially Private LoRA Fine-Tuning of Causal Language Models

Walk through how to fine-tune a GPT-style causal language model on a small generation task with differential privacy, using parameter-efficient LoRA adapters. We compare a non-private LoRA baseline against the DP-LoRA variant and report BLEU, perplexity, peak memory, and throughput for each.

Tracking: [meta-pytorch/opacus#827](https://github.com/meta-pytorch/opacus/issues/827)

## What you'll get out of this notebook

1. A working recipe for combining `opacus`, `peft`, and HuggingFace `transformers` on a causal LM
2. The three device/training-mode ordering patterns that prevent silent corruption (see [#820](https://github.com/meta-pytorch/opacus/issues/820))
3. A concrete sense of the utility cost of DP at a fixed privacy budget on this task
4. Honest notes on what does and does not work yet for full-finetune DP on GPT-2 (out of scope for this tutorial, recipe documented at the end)

## Prerequisites

- Familiarity with the standard DP-SGD setup (`PrivacyEngine`, `make_private_with_epsilon`). The opacus [Building text classifier tutorial](https://github.com/meta-pytorch/opacus/blob/main/tutorials/building_text_classifier.ipynb) is a good warm-up.
- Some prior exposure to LoRA. The [original paper](https://arxiv.org/abs/2106.09685) is short and worth reading.

## Environment and runtime

- Single GPU is sufficient (Kaggle T4 used here)
- Roughly 15 minutes end-to-end on T4 at the settings below (most of it is the 2000-step training pass)
- Pinned versions: `opacus>=1.6.0`, `peft>=0.18,<0.19`, `transformers>=5.0`

## Why combine DP-SGD with LoRA?

DP-SGD adds calibrated Gaussian noise to the per-sample gradient sum at each step. The noise scale grows with the L2 sensitivity of the per-sample gradient (the clipping threshold), so a model with fewer trainable parameters effectively reduces the noise injected into the *learned* parameters. LoRA is a natural fit: it constrains updates to a small low-rank subspace alongside the frozen base weights. In our experiments below, LoRA trains roughly **0.47%** of the GPT-2-small parameters and still reaches a respectable BLEU on the E2E NLG benchmark, both with and without DP.

The combination also lines up cleanly with how practitioners deploy DP today. Sensitive training data warrants a real privacy guarantee; production training budgets warrant parameter-efficient methods. DP-LoRA is the intersection.

## The task: E2E NLG

[E2E NLG](https://arxiv.org/abs/1706.09254) is a small structured-data-to-text generation task originally from the 2017 E2E challenge. Inputs are slot-value meaning representations such as `name[The Vaults], eatType[pub], priceRange[more than £30]`; outputs are short natural-language descriptions. The dataset is small enough to fine-tune in minutes on a single GPU but rich enough that BLEU separates a trained model from chance.

It is also the standard benchmark used in the [DiSK paper](https://arxiv.org/abs/2410.03883) and adjacent DP-NLP work, which makes results here comparable to the published literature.

## 1. Install pinned versions

`peft>=0.18` requires `transformers>=5.0`. The `opacus>=1.6.0` pin is for the version we developed against; older opacus may also work but the integration patterns below assume 1.6+.

In [1]:
!pip install --quiet \
    'opacus>=1.6.0' \
    'peft>=0.18,<0.19' \
    'transformers>=5.0' \
    'accelerate' \
    'datasets' \
    'evaluate' \
    'sacrebleu'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 8.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but

## 2. Sanity-check the environment

If CUDA reports unavailable, enable a GPU accelerator before proceeding. The assertions guard against pip silently resolving to incompatible versions.

In [2]:
import torch
import opacus
import peft
import transformers
import datasets

print(f'torch        = {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
print(f'opacus       = {opacus.__version__}')
print(f'peft         = {peft.__version__}')
print(f'transformers = {transformers.__version__}')
print(f'datasets     = {datasets.__version__}')

if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')


torch        = 2.10.0+cu128  (CUDA: True)
opacus       = 1.6.0
peft         = 0.18.1
transformers = 5.0.0
datasets     = 4.8.5
device       = Tesla T4


## 3. Imports and device

Standard imports plus a seed for partial reproducibility (note: data-loader shuffling and DP noise are still stochastic across runs).

In [3]:
import math
import time
from dataclasses import dataclass, field
from typing import Optional

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from opacus import PrivacyEngine
from datasets import load_dataset
import evaluate

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


Using device: cuda


## 4. Load E2E NLG

The HuggingFace Hub mirrors for E2E NLG (`tuetschek/e2e_nlg`, `GEM/e2e_nlg`) both ship Python loading scripts, which `datasets>=3.0` no longer supports. The dataset is just two CSVs in the upstream repo, so we load them directly with `pandas` and wrap as a `DatasetDict`. Schema after rename: `meaning_representation` (input), `target` (single human reference).

The dev split contains multiple references per MR represented as multiple rows; we treat each row as an independent training example here. A higher-fidelity BLEU evaluation would aggregate references per MR; see the follow-up section at the end.

In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

E2E_TRAIN_URL = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/trainset.csv'
E2E_DEV_URL   = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/devset.csv'

df_train = pd.read_csv(E2E_TRAIN_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})
df_val   = pd.read_csv(E2E_DEV_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})

ds = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'validation': Dataset.from_pandas(df_val),
})
print(ds)
print()
print('--- Sample (train) ---')
print('MR:    ', ds['train'][0]['meaning_representation'])
print('Target:', ds['train'][0]['target'])


DatasetDict({
    train: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 42061
    })
    validation: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 4672
    })
})

--- Sample (train) ---
MR:     name[The Vaults], eatType[pub], priceRange[more than £30], customer rating[5 out of 5], near[Café Adriatic]
Target: The Vaults pub near Café Adriatic has a 5 star rating.  Prices start at £30.


### Subsample validation for fast eval

Use the full train split (about 42K rows). Subsample the validation split to 200 rows so the BLEU sweep across configurations stays under a couple of minutes.

In [5]:
# Pass 2b: use the full E2E NLG train set (~42K) for production. Validation
# stays subsampled (200) to keep BLEU eval fast — full eval is a Phase 3 polish.
VAL_EVAL_SIZE = 200

ds_train_small = ds['train']                                                 # full train
ds_val_small   = ds['validation'].shuffle(seed=42).select(range(VAL_EVAL_SIZE))
print(f'Production train: {len(ds_train_small)}')
print(f'Production val:   {len(ds_val_small)}')


Production train: 42061
Production val:   200


## 5. Tokenize for causal-LM training

Format each row as `'{MR} -> {target}'` and tokenize with GPT-2's tokenizer to a fixed length. We set `labels = input_ids` (predict every position, including the prompt portion). This trains the model slightly differently from a more typical "loss only on the target" setup, but it sidesteps an interaction between `-100`-masked labels, padding, and opacus's per-sample-gradient tracking. See the safety-patterns section below for the full rationale.

In [6]:
MODEL_NAME = 'gpt2'
MAX_SEQ_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

PROMPT_TEMPLATE = '{mr} ->'

def tokenize_example(example):
    """Format as 'MR -> target', tokenize, pad to fixed length.

    NOTE: labels = input_ids (no -100 prompt masking). The model trains on the
    entire sequence, not just the target portion. This is slightly less
    sample-efficient than target-only training, but it sidesteps an opacus
    per-sample-gradient shape-mismatch issue triggered by -100 + padding + LoRA.
    Phase 2 can revisit this with a custom collator if target-only training
    significantly improves BLEU.
    """
    prompt = PROMPT_TEMPLATE.format(mr=example['meaning_representation'])
    target = ' ' + example['target'] + tokenizer.eos_token
    full_text = prompt + target

    enc = tokenizer(
        full_text,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        truncation=True,
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': enc['input_ids'],
    }

ds_train_tok = ds_train_small.map(tokenize_example, remove_columns=ds_train_small.column_names)
ds_val_tok = ds_val_small.map(tokenize_example, remove_columns=ds_val_small.column_names)
ds_train_tok.set_format('torch')
ds_val_tok.set_format('torch')
print(f'Tokenized train: {ds_train_tok}')
print(f'Tokenized val:   {ds_val_tok}')


06/25/2026 05:04:00:WARNING:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/42061 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 42061
})
Tokenized val:   Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


## 6. The three safety patterns

These are the device-placement and training-state orderings that prevent silent corruption when combining HuggingFace + PEFT + opacus on a causal LM. Each of them addresses a real failure mode that we ran into while building this tutorial; we document them inline so readers do not have to rediscover them.

### Pattern A: `model.to(device)` before `get_peft_model()`

With newer PEFT (`>=0.18`), accelerate-style lazy device handling can leave parts of the model on CPU when opacus walks `add_hooks()`. The symptoms are subtle: training loss looks reasonable, the privacy accountant ticks, but LoRA weights never update. We confirmed this empirically across three independent setups (CPU bisect across peft 0.13.2 → 0.18.1, Kaggle T4, RTX 5090) in [opacus#820](https://github.com/meta-pytorch/opacus/issues/820). The safe order is:

```python
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)                # ← move base model to CUDA FIRST
model = get_peft_model(model, config)   # then apply LoRA (LoRA params land on CUDA too)
# then PrivacyEngine.make_private(...)
```

### Pattern B: `model.train()` before `make_private_with_epsilon()`

opacus's `ModuleValidator` (1.6.0+) raises `IllegalModuleConfigurationError("Model needs to be in training mode")` if the model is not in train mode when `make_private_with_epsilon` runs. `get_peft_model()` puts the model in eval mode by default, so an explicit `model.train()` call between PEFT wrapping and opacus wrapping is required.

### Pattern C: `poisson_sampling=False`

opacus's default Poisson sampling occasionally produces an empty batch (probability around `e^(-batch_size)` per step). GPT-2's forward pass calls `attention_mask.view(batch_size, -1)`, which fails for a zero-element tensor because the `-1` dimension is ambiguous. Setting `poisson_sampling=False` switches to uniform-without-replacement, a valid DP-SGD variant (this is what the original [Abadi 2016](https://arxiv.org/abs/1607.00133) paper uses), with deterministic batch sizes and no empty-batch edge case. The accountant handles both regimes correctly.

## 7. Run configuration

Encapsulate the per-run hyperparameters in a small dataclass so the same training loop can drive all configurations.

In [7]:
@dataclass
class RunConfig:
    name: str
    lora: bool
    dp: bool
    lr: float = 1e-4
    batch_size: int = 8
    max_steps: int = 50           # SCAFFOLD; production will be much larger
    target_epsilon: float = 8.0
    target_delta: float = 1e-5
    max_grad_norm: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32

    @property
    def is_full(self) -> bool:
        return not self.lora


## 8. Build the model for a given configuration

Single entry point that handles base-model loading, optional LoRA wrapping, and the Pattern A ordering. The `cfg.dp and not cfg.lora` branch applies `ModuleValidator.fix()` to swap GPT-2's `transformers.Conv1D` modules for `nn.Linear`; this is required when the DP-full path is enabled (which we do not enable in this tutorial — see the follow-up section).

In [8]:
def build_model_for_config(cfg: RunConfig):
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    # DP-full path: swap GPT-2's transformers.Conv1D modules with nn.Linear so opacus's
    # per-sample-gradient hooks attach correctly. opacus has a registered fix for Conv1D.
    # No-op for the LoRA path (LoRA wraps Conv1D with its own A/B Linear modules, which
    # opacus already handles).
    if cfg.dp and not cfg.lora:
        from opacus.validators import ModuleValidator
        model = ModuleValidator.fix(model)
        print(f'[{cfg.name}] Applied ModuleValidator.fix() (Conv1D -> nn.Linear)')

    model = model.to(device)  # <-- BEFORE PEFT, per #820

    if cfg.lora:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=0.0,
            target_modules=['c_attn'],
        )
        model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'[{cfg.name}] trainable={trainable:,} ({100*trainable/total:.2f}% of {total:,})')
    return model


## 9. The shared training function

Same loop for all configurations. The `cfg.dp` block adds the `PrivacyEngine`, applies the three safety patterns, and uses `poisson_sampling=False` per Pattern C. Memory and throughput are tracked across the run; the final epsilon is read from the accountant at the end. The per-step skip on empty batches is a defensive guard that should never actually fire with `poisson_sampling=False`.

In [9]:
def train_one_run(cfg: RunConfig, train_ds, val_ds):
    print(f'\n=== Training: {cfg.name} ===')
    model = build_model_for_config(cfg)
    model.train()  # required: opacus validator (>=1.6.0) checks model.training before make_private_with_epsilon

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr,
    )
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)

    privacy_engine = None
    if cfg.dp:
        privacy_engine = PrivacyEngine(accountant='rdp')
        grad_sample_mode = 'functorch' if not cfg.lora else 'hooks'
        print(f'[{cfg.name}] grad_sample_mode={grad_sample_mode}')
        # poisson_sampling=False uses uniform-without-replacement (deterministic batch sizes).
        # Default Poisson sampling can produce 0-sample batches that GPT-2's forward pass
        # cannot handle (reshape of (0, -1) is ambiguous). Uniform sampling is a documented
        # opacus mode and a standard DP-SGD variant — see opacus.PrivacyEngine docs.
        model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader,
            target_epsilon=cfg.target_epsilon,
            target_delta=cfg.target_delta,
            epochs=max(1, cfg.max_steps // (len(train_ds) // cfg.batch_size + 1)),
            max_grad_norm=cfg.max_grad_norm,
            grad_sample_mode=grad_sample_mode,
            poisson_sampling=False,
        )
        print(f'[{cfg.name}] noise_multiplier={optimizer.noise_multiplier:.4f}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()
    tokens_processed = 0

    model.train()
    step = 0
    loss_history = []
    while step < cfg.max_steps:
        for batch in train_loader:
            if step >= cfg.max_steps:
                break
            # Defensive: skip any empty batches (shouldn't happen with poisson_sampling=False,
            # but harmless guard)
            if batch['input_ids'].numel() == 0:
                continue
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_history.append(loss.item())
            tokens_processed += batch['input_ids'].numel()
            if step % 100 == 0:
                print(f'  step {step:4d}  loss={loss.item():.4f}')
            step += 1

    t_total = time.perf_counter() - t_start
    peak_mem_gb = (torch.cuda.max_memory_allocated() / 1024**3) if torch.cuda.is_available() else 0.0
    final_epsilon = privacy_engine.get_epsilon(delta=cfg.target_delta) if cfg.dp else float('inf')

    return {
        'config': cfg,
        'model': model,
        'mean_loss': sum(loss_history) / len(loss_history),
        'final_loss': loss_history[-1],
        'tokens_per_sec': tokens_processed / t_total,
        'peak_mem_gb': peak_mem_gb,
        'wall_clock_sec': t_total,
        'epsilon': final_epsilon,
        'steps_completed': step,
    }


## 10. Evaluation: BLEU plus perplexity

BLEU is computed by greedy generation from the prompt portion of each validation example (decoded against the held-out target). Perplexity uses the standard `exp(mean(eval_loss))` formulation. Both are reported per configuration.

In [10]:
bleu_metric = evaluate.load('sacrebleu')

# Re-tokenize to recover prompt boundary at eval time (since labels no longer carry it)
def _prompt_token_len(mr_text):
    prompt = PROMPT_TEMPLATE.format(mr=mr_text)
    return len(tokenizer(prompt, add_special_tokens=False)['input_ids'])

def evaluate_run(model, val_ds_raw, val_ds_tok, max_eval_examples=50, max_new_tokens=40):
    """Compute BLEU on generated text + perplexity on val loss.
    val_ds_raw provides the raw MR + target for prompt boundary lookup.
    """
    gen_model = model._module if hasattr(model, '_module') else model
    gen_model.eval()

    preds, refs = [], []
    val_losses = []

    n = min(max_eval_examples, len(val_ds_tok))
    with torch.no_grad():
        for i in range(n):
            ex_tok = val_ds_tok[i]
            ex_raw = val_ds_raw[i]
            input_ids = ex_tok['input_ids'].unsqueeze(0).to(device)
            attention_mask = ex_tok['attention_mask'].unsqueeze(0).to(device)
            labels = ex_tok['labels'].unsqueeze(0).to(device)

            # Perplexity
            out = gen_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_losses.append(out.loss.item())

            # Generate from the prompt portion only
            prompt_len = _prompt_token_len(ex_raw['meaning_representation'])
            prompt_ids = input_ids[0, :prompt_len].unsqueeze(0)
            gen = gen_model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            pred = tokenizer.decode(gen[0, prompt_len:], skip_special_tokens=True).strip()
            ref = ex_raw['target']
            preds.append(pred)
            refs.append([ref])

    bleu = bleu_metric.compute(predictions=preds, references=refs)['score']
    perplexity = math.exp(sum(val_losses) / len(val_losses))
    return {'bleu': bleu, 'perplexity': perplexity, 'n_eval': len(preds)}


## 11. Configurations to compare

Two configurations, both LoRA-based, identical in every respect except whether the `PrivacyEngine` is attached. The `lr=5e-4` for the DP variant is intentionally higher than the non-DP `1e-4`: DP-SGD's per-step update is dampened by gradient clipping and Gaussian noise, so a higher learning rate is needed to make comparable progress. Empirically, `lr=1e-4` produced BLEU near zero for the DP variant in our early runs.

DP-full (no LoRA) is intentionally left out of this tutorial; see the follow-up section for what is needed to enable it.

In [11]:
# Pass 2b retry: DP-LoRA lr bumped 1e-4 -> 5e-4. The 1e-4 setting (same as non-DP)
# produced BLEU ≈ 0.06 despite the model reaching PPL ≈ 6.66 — classic DP-SGD
# symptom that the effective per-step update is too small under noise+clipping.
# Higher lr compensates. If 5e-4 still doesn't break BLEU, try 1e-3 or a small sweep.
CONFIGS = [
    RunConfig(name='non-DP LoRA', lora=True, dp=False, lr=1e-4, batch_size=8, max_steps=2000),
    RunConfig(name='DP LoRA',     lora=True, dp=True,  lr=5e-4, batch_size=8, max_steps=2000, target_epsilon=8.0),
    # RunConfig(name='DP full',   lora=False, dp=True, ...),  # deferred — see note in #11
]


## 12. Run both configurations

Sequential. Memory is freed between runs to avoid cross-config peak memory accounting confusion.

In [12]:
results = []
for cfg in CONFIGS:
    train_result = train_one_run(cfg, ds_train_tok, ds_val_tok)
    eval_result = evaluate_run(train_result['model'], ds_val_small, ds_val_tok)
    results.append({**train_result, **eval_result})
    print(f'[{cfg.name}] BLEU={eval_result["bleu"]:.2f}  PPL={eval_result["perplexity"]:.2f}')
    del train_result['model']
    torch.cuda.empty_cache() if torch.cuda.is_available() else None



=== Training: non-DP LoRA ===


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

[DP LoRA] trainable=589,824 (0.47% of 125,029,632)
[DP LoRA] grad_sample_mode=hooks


/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


[DP LoRA] noise_multiplier=0.3717


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


  step    0  loss=8.8156
  step  100  loss=6.4995
  step  200  loss=2.7243
  step  300  loss=2.2852
  step  400  loss=1.8140
  step  500  loss=1.5359
  step  600  loss=1.5013
  step  700  loss=1.8234
  step  800  loss=1.3608
  step  900  loss=1.5607
  step 1000  loss=1.4962
  step 1100  loss=1.2945
  step 1200  loss=1.2772
  step 1300  loss=1.2257
  step 1400  loss=1.2012
  step 1500  loss=1.0384
  step 1600  loss=1.2010
  step 1700  loss=1.2298
  step 1800  loss=0.9470
  step 1900  loss=1.1885
[DP LoRA] BLEU=18.24  PPL=2.71


## 13. Comparison table

In [13]:
def fmt_eps(e):
    return 'no DP' if e == float('inf') else f'{e:.2f}'

print(f'{"Config":<14} | {"BLEU":>6} | {"PPL":>7} | {"tok/s":>8} | {"peak GB":>7} | {"sec":>6} | {"ε":>6}')
print('-' * 75)
for r in results:
    print(f'{r["config"].name:<14} | {r["bleu"]:>6.2f} | {r["perplexity"]:>7.2f} | '
          f'{r["tokens_per_sec"]:>8.0f} | {r["peak_mem_gb"]:>7.2f} | '
          f'{r["wall_clock_sec"]:>6.1f} | {fmt_eps(r["epsilon"]):>6}')


Config         |   BLEU |     PPL |    tok/s | peak GB |    sec |      ε
---------------------------------------------------------------------------
non-DP LoRA    |  24.72 |    1.60 |     4679 |    2.17 |  437.7 |  no DP
DP LoRA        |  18.24 |    2.71 |     4599 |    2.65 |  445.3 |   7.08


## 14. Reading the results

Sample run on Kaggle T4 (your numbers will vary slightly with random seeds, hardware, and library minor versions):

| Config | BLEU | PPL | tok/s | Peak GB | Time | ε |
|---|---|---|---|---|---|---|
| non-DP LoRA | 24.72 | 1.60 | 4679 | 2.17 | 438 s | no DP |
| DP LoRA | 18.24 | 2.71 | 4599 | 2.65 | 445 s | 7.08 |

The non-DP LoRA baseline scores BLEU 24.72 on E2E NLG after 2000 steps while training only 589K of GPT-2-small's 125M parameters (about 0.47%). That is the parameter-efficiency story that motivates LoRA in the first place: most of the useful signal can be captured in a small low-rank perturbation, and the rest of the network does not need to move.

DP costs roughly 26% relative BLEU at ε ≈ 7 here, going from 24.72 down to 18.24. Perplexity moves from 1.60 to 2.71. The model still clearly learns the conditional generation task; the noise from DP-SGD shifts utility downward but does not collapse it. The 26% figure is in the ballpark of what comparable papers report at similar privacy budgets on this task.

Memory overhead from opacus is modest at LoRA's parameter scale. About 22% more peak GPU memory (2.17 GB to 2.65 GB) covers the per-sample-gradient accumulation. This stays comfortably within a T4's 16 GB envelope and would not constrain a typical fine-tuning workflow.

Throughput is essentially unchanged. Both configurations hit roughly 4600 tokens per second; opacus is not a throughput bottleneck for LoRA at this size.

**Learning rate matters more for DP.** The 5× higher lr for the DP variant is not optional. Leave both configurations at `lr=1e-4` and the DP variant produces BLEU near zero despite reaching reasonable perplexity. The intuition: gradient clipping and Gaussian noise both shrink the effective per-step update, so the optimizer needs more aggressive learning to make comparable progress per step. This is consistent with the DP-NLP literature and worth surfacing explicitly in any production setup.

## 15. When to reach for DP-LoRA

A practical heuristic, given the numbers above:

Use DP-LoRA when the training data is sensitive enough to warrant a real privacy guarantee and the downstream task tolerates a meaningful utility drop relative to a non-DP baseline. The 27% relative BLEU cost we observe is in the ballpark reported elsewhere in the DP-NLP literature at comparable ε; it is the price of admission for the formal privacy guarantee.

Consider full DP fine-tuning instead of DP-LoRA when LoRA's restricted parameter subspace is the bottleneck (not the privacy budget) and you have compute headroom to handle opacus's per-sample-gradient memory overhead at the full model scale. The full-finetune path on GPT-2 specifically also needs the engineering steps in the follow-up section below.

## 16. Follow-up work

### Enabling DP-full fine-tuning on GPT-2

DP fine-tuning of GPT-2 with all 125M parameters trainable is out of scope for this tutorial. The straightforward setup runs into a per-sample-gradient shape mismatch in opacus's `clip_and_accumulate` step that neither `ModuleValidator.fix()` (Conv1D → Linear swap) nor `grad_sample_mode='functorch'` (vmap-based per-sample grads) resolved in our testing. The most likely root cause is GPT-2's tied input embedding and output projection (`transformer.wte.weight` is the same tensor as `lm_head.weight`); opacus's hook-based accumulation appears to double-count or miscount across the two module sites.

A reasonable engineering recipe to restore DP-full as a follow-up PR:

1. Explicitly untie the embedding weight: `model.lm_head.weight = nn.Parameter(model.transformer.wte.weight.data.clone())`
2. Apply `opacus.validators.ModuleValidator.fix(model)` to swap `transformers.Conv1D` modules for `nn.Linear`
3. Use `grad_sample_mode='functorch'` for additional robustness against custom module types
4. Reduce `batch_size` to fit T4 memory (functorch's per-sample-grad path is 3 to 4× heavier than the default hooks path for GPT-2-full)

### Other polishes worth doing

- Full validation set (about 4K examples) for higher-confidence BLEU
- Multi-reference BLEU using all human references per MR (the E2E NLG release includes 5 to 8 refs per input)
- 2 to 3 seeds per configuration to quantify variance
- Small HP sweep around `lr`, `max_grad_norm`, and `target_epsilon` to pick "reasonable" settings rigorously
- Loss-only-on-target labels (with `-100` masking on the prompt portion) once the opacus + LoRA per-sample-gradient interaction with masked labels is resolved

### References

- [opacus#820](https://github.com/meta-pytorch/opacus/issues/820) — the device-placement-ordering issue that motivates Pattern A
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [Deep Learning with Differential Privacy (Abadi et al. 2016)](https://arxiv.org/abs/1607.00133)
- [E2E NLG Challenge dataset](https://arxiv.org/abs/1706.09254) and the upstream [GitHub repo](https://github.com/tuetschek/e2e-dataset)
- [DiSK: Differentially Private Optimizer with Simplified Kalman Filter (Zhang et al. 2024)](https://arxiv.org/abs/2410.03883), which uses E2E NLG as part of its benchmark suite